In [1]:
# For older gpu
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# For newer
# !pip install torch

In [32]:
import torch
import time
import math
import pandas as pd
import random
from tokenizer import load_rules_tokens, Tokenizer
from torch.utils.data import RandomSampler
from BPETokenizer import BPETokenizer
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import random_split

In [3]:
PADDING_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
UNK_TOKEN = 3

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device of type: {device.type}')

Using device of type: cuda


In [5]:
def format_time(seconds):
    mins = int(seconds / 60)
    secs = int(seconds % 60)
    return f"{mins}m {secs}s"

In [6]:
def load_data(filename, num_samples=10000):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')

    en_texts = []
    fr_texts = []
    if isinstance(num_samples, float):
        assert 0. < num_samples <= 1., 'For float num_samples, it has to be in (0, 1]'
        num_samples = int(len(lines)*num_samples)
    
    assert isinstance(num_samples, int), f'num_samples is of wrong type: {type(num_samples)}'
    assert num_samples >= 0, 'num_samples can\'t be negative'

    for line in lines[:num_samples]:
        parts = line.split('\t')
        if len(parts) < 2: continue
        
        en_texts.append(parts[0])
        fr_texts.append(parts[1])
    
    return en_texts, fr_texts

In [7]:
def load_data_csv(filename) -> tuple[list, list]:
    data = pd.read_csv(filename).dropna()
    return data['en'].astype(str).to_list(), data['fr'].astype(str).to_list()

In [8]:
def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs_padded = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=PADDING_TOKEN)
    targets_padded = nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=PADDING_TOKEN)
    
    return inputs_padded, targets_padded

In [9]:
class Vocab:
    def __init__(self):
        self.token2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UKN>": 3}
        self.index2char = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UKN>"}
        self.n_chars = 4
    
    def add_tokens(self, tokens):
        for token in tokens:
            if token not in self.token2index:
                self.token2index[token] = self.n_chars
                self.index2char[self.n_chars] = token
                self.n_chars += 1

In [10]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, embedding_dim=32, drop_out=0.2):
        super().__init__()
        self.embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=num_layers, 
                            dropout=drop_out if num_layers > 1 else 0, batch_first=True, bidirectional=True)
    
    def forward(self, X):
        embs = self.embedder(X)
        out, hidden = self.lstm(embs)
        return out, hidden

In [11]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1, embedding_dim=32, drop_out=0.2):
        super().__init__()
        self.embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, 
                            batch_first=True, dropout=drop_out if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, X, hidden):
        embs = self.embedder(X)
        output, hidden = self.lstm(embs, hidden)
        preds = self.out(output)
        return preds, hidden

In [12]:
class TranslateDataset(Dataset):
    def __init__(self, X, Y, in_vocab, out_vocab):
        self.X = X
        self.Y = Y
        self.in_vocab = in_vocab
        self.out_vocab = out_vocab
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        X, y = self.X[idx], self.Y[idx]
        input_indices = [self.in_vocab.token2index[t] for t in X] + [EOS_TOKEN]
        target_indices = [SOS_TOKEN] + [self.out_vocab.token2index[t] for t in y] + [EOS_TOKEN]
        return torch.tensor(input_indices), torch.tensor(target_indices)

In [33]:
class Seq2Seq(nn.Module):
    def __init__(self, in_vocab, out_vocab, hidden_size, num_layers=1, embedding_dim=32, drop_out=0.2, device='cuda'):
        super().__init__()
        self.in_vocab = in_vocab
        self.out_vocab = out_vocab
        self.device = device
        
        self.encoder = Encoder(in_vocab.n_chars, hidden_size, num_layers, embedding_dim, drop_out)
        self.decoder = Decoder(out_vocab.n_chars, hidden_size*2, num_layers, embedding_dim, drop_out)
        self.to(device)
    
    def _bridge_hidden(self, hidden):
        h, c = hidden
        num_layers = self.encoder.lstm.num_layers
        batch_size = h.size(1)
        hidden_size = self.encoder.lstm.hidden_size
        
        h = h.reshape(num_layers, 2, batch_size, hidden_size)
        c = c.reshape(num_layers, 2, batch_size, hidden_size)
        
        h = torch.cat((h[:, 0, :, :], h[:, 1, :, :]), dim=-1)
        c = torch.cat((c[:, 0, :, :], c[:, 1, :, :]), dim=-1)
        
        return h, c
        
    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.size(0)
        target_len = target.size(1)
        out_vocab_size = self.out_vocab.n_chars
        
        outputs = torch.zeros(batch_size, target_len - 1, out_vocab_size).to(self.device)
        
        _, hidden = self.encoder(source)
        
        hidden = self._bridge_hidden(hidden)
        
        if teacher_forcing_ratio == 1.0:
            predictions, _ = self.decoder(target[:, :-1], hidden)
            return predictions
            
        decoder_input = target[:, 0].unsqueeze(1) 
        
        for t in range(0, target_len - 1):
            predictions, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = predictions.squeeze(1)

            top1 = predictions.argmax(dim=-1)
            
            use_teacher_forcing = random.random() < teacher_forcing_ratio
            decoder_input = target[:, t+1].unsqueeze(1) if use_teacher_forcing else top1
            
        return outputs

    def save_model(self, path="best_seq2seq.pth", verbose=True):
        torch.save(self.state_dict(), path)
        if verbose: print(f"-> Model saved to {path}")

    def load_model(self, path="best_seq2seq.pth", verbose=True):
        self.load_state_dict(torch.load(path, map_location=self.device))
        if verbose: print(f"-> Model loaded from {path}")

    def train_model(self, X, y, epochs=32, patience=5, min_delta=0.01, lr=0.001, batch_size=64, 
                    tf_start=1.0, tf_end=0.0, tf_decay_start=5, tf_decay_end=20, early_stopping_start=15):
        
        dataset = TranslateDataset(X, y, self.in_vocab, self.out_vocab)
        val_size = int(0.1 * len(dataset))
        train_size = len(dataset) - val_size
        train_ds, val_ds = random_split(dataset, [train_size, val_size])
        
        num_samples = int(0.7 * len(train_ds))
        
        train_sampler = RandomSampler(
            train_ds, 
            replacement=True, 
            num_samples=num_samples
        )
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
        
        criterion = nn.CrossEntropyLoss(ignore_index=PADDING_TOKEN) 
        optimizer = torch.optim.AdamW(self.parameters(), lr=lr)
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        print(f"{'='*60}\nStarting Training\n{'='*60}")
        
        for epoch in range(1, epochs + 1):
            start_time = time.time()
            
            if epoch <= tf_decay_start:
                tf_ratio = tf_start
            elif epoch >= tf_decay_end:
                tf_ratio = tf_end
            else:
                decay_progress = (epoch - tf_decay_start) / max(1, (tf_decay_end - tf_decay_start))
                tf_ratio = tf_start - (tf_start - tf_end) * decay_progress
            
            self.train()
            train_loss = 0
            
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                optimizer.zero_grad()
                
                predictions = self(inputs, targets, teacher_forcing_ratio=tf_ratio)
                
                loss = criterion(
                    predictions.reshape(-1, self.out_vocab.n_chars), 
                    targets[:, 1:].reshape(-1)
                )
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()
                
            avg_train_loss = train_loss / len(train_loader)
            
            self.eval()
            val_loss = 0
            
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    
                    predictions = self(inputs, targets, teacher_forcing_ratio=0.0)
                    
                    loss = criterion(
                        predictions.reshape(-1, self.out_vocab.n_chars), 
                        targets[:, 1:].reshape(-1)
                    )
                    val_loss += loss.item()
                    
            avg_val_loss = val_loss / len(val_loader)
            epoch_mins_secs = format_time(time.time() - start_time)
            
            print(f"Epoch: {epoch:02d}/{epochs} | Time: {epoch_mins_secs} | TF Ratio: {tf_ratio:.2f}")
            print(f"\tTrain Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

            if avg_val_loss < (best_val_loss - min_delta):
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                self.save_model("best_model.pth", verbose=False)
            else:
                if epoch >= early_stopping_start:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        print(f"\n[!] Early stopping triggered. No improvement for {patience} epochs.")
                        print("-> Loading best model weights back into memory...")
                        self.load_model("best_model.pth", verbose=False)
                        break
                else:
                    epochs_no_improve = 0

In [14]:
# en_tokenizer = Tokenizer.load('./en_bpe_model.json')
# fr_tokenizer = Tokenizer.load('./fr_bpe_model.json')

In [15]:
en_tokenizer = BPETokenizer.load('./en_bpe_model.json')
fr_tokenizer = BPETokenizer.load('./fr_bpe_model.json')

In [16]:
# en_texts, fr_texts = load_data('./', 1.)
en_texts, fr_texts = load_data_csv('./en-fr-1-mil.csv')

In [19]:
en_tokenized = en_tokenizer.encode_batch(en_texts)
fr_tokenized = fr_tokenizer.encode_batch(fr_texts)

In [22]:
in_vocab = Vocab()
in_vocab.add_tokens(en_tokenizer.tokens)

out_vocab = Vocab()
out_vocab.add_tokens(fr_tokenizer.tokens)

In [ ]:
model = Seq2Seq(in_vocab, out_vocab, hidden_size=128, num_layers=4, embedding_dim=42, drop_out=0, device=device)

# model.load_model('./best_model.pth')

In [40]:
model.train_model(
    en_tokenized, 
    fr_tokenized, 
    batch_size=32, 
    epochs=40, 
    lr=0.001,
    
    # --- Teacher Forcing Rules ---
    tf_start=0.0,           # Start with 100% teacher forcing
    tf_decay_start=5,       # Keep it at 100% until epoch 5
    tf_decay_end=20,        # Smoothly lower it, hitting 0% at epoch 20
    tf_end=0.0,             # Keep it at 0% from epoch 20 to 40
    
    # --- Early Stopping Rules ---
    patience=5,             # Stop if no improvement for 5 epochs...
    early_stopping_start=15 # ...but don't even start checking until epoch 15
)

Starting Training


OutOfMemoryError: CUDA out of memory. Tried to allocate 760.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 186.75 MiB is free. Including non-PyTorch memory, this process has 3.48 GiB memory in use. Of the allocated memory 3.17 GiB is allocated by PyTorch, and 210.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import gc
import torch

# Force Python garbage collection
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()